In [ ]:
# step3_keyword_detection.py

import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"

# === Read data from Step 2 ===
df = pd.read_csv(input_path)

android_present = []
android_in_readme = []
confidence = []

for i, row in df.iterrows():
    if row["status"] != "pass":
        android_present.append("N/A")
        android_in_readme.append("N/A")
        confidence.append("N/A")
        continue

    found = False
    in_readme = False

    # Check in name, description, topics
    name = str(row.get("name", "")).lower()
    desc = str(row.get("description", "")).lower()
    topics = str(row.get("topics", "")).lower()
    if "android" in name or "android" in desc or "android" in topics:
        found = True

    # Check README
    repo = row["full_name"]
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    response = requests.get(readme_url, headers=get_headers())
    if response.status_code == 200:
        content = response.json().get("content", "")
        decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
        if "android" in decoded:
            in_readme = True

    is_android = found or in_readme
    android_present.append("yes" if is_android else "no")
    android_in_readme.append("yes" if in_readme else "no")
    confidence.append("low" if in_readme and not found else ("high" if is_android else "none"))

    if i % 100 == 0:
        print(f"🔍 Checked {i+1} repos...")

df["android_keyword"] = android_present
df["android_in_readme"] = android_in_readme
df["confidence"] = confidence
df.to_csv(output_path, index=False)
print(f"✅ Step 3 complete. Saved to: {output_path}")
